In [1]:
import numpy as np
from sklearn.linear_model import RidgeClassifierCV
from sklearn.preprocessing import StandardScaler

from aeon.classification import BaseClassifier
from aeon.classification.convolution_based._hydra import _SparseScaler
from aeon.transformations.collection.convolution_based import MultiRocket
from aeon.transformations.collection.convolution_based._hydra import HydraTransformer
from aeon.utils.validation import check_n_jobs
from aeon.transformations.collection.interval_based import QUANTTransformer
import os
import random
from itertools import product
from time import perf_counter
from aeon.transformations.collection.shapelet_based import (
    RandomDilatedShapeletTransform,
)
import numpy as np
import polars as pl
from aeon.classification.base import BaseClassifier
from aeon.classification.feature_based import (
    Catch22Classifier,
)
from aeon.transformations.collection.convolution_based import Rocket
from aeon.datasets.tsc_datasets import univariate
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestClassifier,
)
from aeon.classification.convolution_based import MultiRocketHydraClassifier
from aeon.classification.convolution_based import RocketClassifier
from sklearn.metrics import accuracy_score
from aeon.classification.interval_based import QUANTClassifier
from autotsc import utils, models
from aeon.transformations.collection import Normalizer
from aeon.benchmarking.results_loaders import get_available_estimators
from aeon.benchmarking.results_loaders import get_estimator_results
from tqdm import tqdm


2025-12-03 13:51:26.465961: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
dataset = 'Computers'
X_train, y_train, X_test, y_test = utils.load_dataset(dataset)

In [13]:
classifiers = get_available_estimators(task="classification")['classification'].to_list()
datasets = ["ACSF1", "ArrowHead", "GunPoint", 'AllGestureWiimoteZ']

results_dict = get_estimator_results(estimators=classifiers, datasets=datasets)
for classfier in classifiers:
    print(classfier, results_dict[classfier]['GunPoint'])

1NN-DTW 0.9066666666666666
Arsenal 1.0
BOSS 1.0
CIF 0.98
CNN 0.9666666666666668
Catch22 0.9666666666666668
DrCIF 0.9933333333333332
EE 0.9533333333333334
FreshPRINCE 0.94
GRAIL 0.86
H-InceptionTime 1.0
HC1 1.0
HC2 1.0
Hydra 1.0
InceptionTime 1.0
LiteTime 1.0
MR 1.0
MR-Hydra 1.0
MiniROCKET 0.9933333333333332
MrSQM 1.0
PF 1.0
QUANT 0.9866666666666668
R-STSF 0.98
RDST 1.0
RISE 0.98
RIST 1.0
ROCKET 1.0
RSF 0.9666666666666668
ResNet 0.9933333333333332
STC 1.0
STSF 0.9066666666666666
ShapeDTW 0.96
Signatures 0.9533333333333334
TDE 1.0
TS-CHIEF 1.0
TSF 0.9666666666666668
TSFresh 0.9533333333333334
WEASEL-1.0 0.9933333333333332
WEASEL-2.0 1.0
cBOSS 1.0


In [4]:
class CrossValidationWrapper(BaseClassifier):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.trained_models_ = []
        self.fit_time_ = []
        self.fit_time_mean_ = None
        self.predict_time_ = []
        self.predict_time_mean_ = None

    def _fit(self, X, y):
        raise NotImplementedError()

    def _predict_proba(self, X):
        predictions = []
        for model in self.trained_models_:
            proba = model.predict_proba(X)
            predictions.append(proba)
        avg_proba = np.mean(predictions, axis=0)
        return avg_proba

    def _predict(self, X):
        probas = self._predict_proba(X)
        predicted_indices = np.argmax(probas, axis=1)
        return self.classes_[predicted_indices]

    def _fit_predict_proba(self, X, y, cv_splits):
        oof_proba = []
        for train_idx, val_idx in tqdm(cv_splits):
            model_clone = clone(self.model)
            X_train, y_train = X[train_idx], y[train_idx]
            X_valid, _ = X[val_idx], y[val_idx]
            model_clone.fit(X_train, y_train)
            self.trained_models_.append(model_clone)
            proba = model_clone.predict_proba(X_valid)
            prob_columns = []
            for idx, p in zip(val_idx, proba):
                d = {
                    'index': idx,
                }
                for scls, prob in zip(model_clone.classes_, p):
                    k = f'proba_class_{scls}'
                    d[k] = prob.item()
                    if k not in prob_columns:
                        prob_columns.append(k)
                oof_proba.append(d)
        return pl.DataFrame(oof_proba).group_by('index').mean().sort('index').select(prob_columns).to_numpy()

    def fit_predict_proba(self, X, y, cv_splits):
        X, y, single_class = self._fit_setup(X, y)
        y_proba = self._fit_predict_proba(X, y, cv_splits)
        self.is_fitted = True
        return y_proba

from aeon.classification.feature_based import Catch22Classifier

folds1 = utils.get_folds(X_train, y_train)
folds2 = utils.get_folds(X_train, y_train)
folds3 = utils.get_folds(X_train, y_train)
folds4 = utils.get_folds(X_train, y_train)
all_folds = folds1 + folds2 + folds3 + folds4

In [5]:
m_catch22 = CrossValidationWrapper(model=Catch22Classifier(n_jobs=-1))
oof_pred_prob_catch22 = m_catch22.fit_predict_proba(X_train, y_train, all_folds)

100%|██████████| 40/40 [00:35<00:00,  1.12it/s]


In [6]:
m_quant = CrossValidationWrapper(model=QUANTClassifier())
oof_pred_prob_quant = m_quant.fit_predict_proba(X_train, y_train, all_folds)

100%|██████████| 40/40 [01:39<00:00,  2.50s/it]


In [7]:
p_catch22 = pl.DataFrame(oof_pred_prob_catch22, schema=list(m_catch22.classes_)).with_columns(
    pl.when(pl.col("1") > pl.col("2"))
      .then(pl.lit("1"))
      .otherwise(pl.lit("2"))
      .alias("label")
)
print(accuracy_score(y_train, p_catch22["label"].to_numpy()))
p_quant = pl.DataFrame(oof_pred_prob_quant, schema=list(m_catch22.classes_)).with_columns(
    pl.when(pl.col("1") > pl.col("2"))
      .then(pl.lit("1"))
      .otherwise(pl.lit("2"))
      .alias("label")
)
print(accuracy_score(y_train, p_quant["label"].to_numpy()))

0.768
0.88


In [8]:
import numpy as np
from sklearn.linear_model import RidgeClassifierCV
from sklearn.preprocessing import StandardScaler

from aeon.classification import BaseClassifier
from aeon.classification.convolution_based._hydra import _SparseScaler
from aeon.transformations.collection.convolution_based import MultiRocket
from aeon.transformations.collection.convolution_based._hydra import HydraTransformer
from aeon.utils.validation import check_n_jobs


class MyMultiRocketHydraClassifier(BaseClassifier):
    _tags = {
        "capability:multivariate": True,
        "capability:multithreading": True,
        "algorithm_type": "convolution",
        "python_dependencies": "torch",
    }

    def __init__(
        self,
        n_kernels: int = 8,
        n_groups: int = 64,
        class_weight=None,
        n_jobs: int = 1,
        random_state=None,
    ):
        self.n_kernels = n_kernels
        self.n_groups = n_groups
        self.class_weight = class_weight
        self.n_jobs = n_jobs
        self.random_state = random_state

        super().__init__()

    def _fit(self, X, y):
        self._n_jobs = check_n_jobs(self.n_jobs)
        self.m_quant = CrossValidationWrapper(model=QUANTClassifier())

        folds1 = utils.get_folds(X, y)
        folds2 = utils.get_folds(X, y)
        folds3 = utils.get_folds(X, y)
        folds4 = utils.get_folds(X, y)
        all_folds = folds1 + folds2 + folds3 + folds4

        XQ = self.m_quant.fit_predict_proba(X, y, all_folds)

        self._transform_hydra = HydraTransformer(
            n_kernels=self.n_kernels,
            n_groups=self.n_groups,
            n_jobs=self._n_jobs,
            random_state=self.random_state,
        )
        Xt_hydra = self._transform_hydra.fit_transform(X)

        self._scale_hydra = _SparseScaler()
        Xt_hydra = self._scale_hydra.fit_transform(Xt_hydra)

        self._transform_multirocket = MultiRocket(
            n_jobs=self._n_jobs,
            random_state=self.random_state,
        )
        Xt_multirocket = self._transform_multirocket.fit_transform(X)

        self._scale_multirocket = StandardScaler()
        Xt_multirocket = self._scale_multirocket.fit_transform(Xt_multirocket)

        Xt = np.concatenate((XQ, Xt_hydra, Xt_multirocket), axis=1)

        self.classifier = RidgeClassifierCV(
            alphas=np.logspace(-3, 3, 10), class_weight=self.class_weight
        )
        self.classifier.fit(Xt, y)

        return self

    def _predict(self, X) -> np.ndarray:
        XQ = self.m_quant._predict_proba(X)
        Xt_hydra = self._transform_hydra.transform(X)
        Xt_hydra = self._scale_hydra.transform(Xt_hydra)

        Xt_multirocket = self._transform_multirocket.transform(X)
        Xt_multirocket = self._scale_multirocket.transform(Xt_multirocket)

        Xt = np.concatenate((XQ, Xt_hydra, Xt_multirocket), axis=1)

        return self.classifier.predict(Xt)

In [9]:
for _ in range(10):
    c1 = MyMultiRocketHydraClassifier(n_jobs=-1)
    c2 = MultiRocketHydraClassifier(n_jobs=-1)
    c1.fit(X_train, y_train)
    c2.fit(X_train, y_train)
    p1 = c1.predict(X_test)
    p2 = c2.predict(X_test)
    acc1 = accuracy_score(y_test, p1)
    acc2 = accuracy_score(y_test, p2)
    print(acc1, acc2)

100%|██████████| 40/40 [00:46<00:00,  1.16s/it]


0.804 0.78


100%|██████████| 40/40 [02:53<00:00,  4.33s/it]


0.784 0.78


  8%|▊         | 3/40 [00:46<09:29, 15.38s/it]


KeyboardInterrupt: 